In [ ]:
import sys
import re
import os
import pandas as pd
import numpy as np
import openpyxl

print('Roll The Dice - Oracle Solution', file=sys.stderr)

# Determine data path
DATA_DIR = '/workspace/data'
if not os.path.exists(os.path.join(DATA_DIR, 'MO16-Round-1-Sec-3-Roll-The-Dice-1.xlsx')):
    for candidate in [
        os.path.join(os.path.dirname(os.path.abspath('.')), 'environment', 'data'),
        os.path.join(os.path.abspath('.'), 'environment', 'data'),
        os.path.join(os.path.abspath('.'), 'data'),
    ]:
        if os.path.exists(os.path.join(candidate, 'MO16-Round-1-Sec-3-Roll-The-Dice-1.xlsx')):
            DATA_DIR = candidate
            break

EXCEL_PATH = os.path.join(DATA_DIR, 'MO16-Round-1-Sec-3-Roll-The-Dice-1.xlsx')
print(f'Data dir: {DATA_DIR}', file=sys.stderr)
print(f'Excel: {EXCEL_PATH}', file=sys.stderr)
print(f'Exists: {os.path.exists(EXCEL_PATH)}', file=sys.stderr)

In [ ]:
# ======================================================
# 1. LOAD AND EXPLORE THE EXCEL FILE
# ======================================================
print('1. Loading Excel file...', file=sys.stderr)

# Try data_only=True first (reads cached formula results)
wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
print(f'  Sheet names: {wb.sheetnames}', file=sys.stderr)

# Find the sheet with dice data - try all sheets
best_ws = None
best_sheet_name = None
for sn in wb.sheetnames:
    ws = wb[sn]
    print(f'  Sheet "{sn}": {ws.max_row} rows x {ws.max_column} cols', file=sys.stderr)
    if ws.max_row and ws.max_row > 100:
        best_ws = ws
        best_sheet_name = sn

if best_ws is None:
    best_ws = wb[wb.sheetnames[0]]
    best_sheet_name = wb.sheetnames[0]

ws = best_ws
print(f'  Using sheet: "{best_sheet_name}"', file=sys.stderr)

# Print first 20 rows for debugging
for row_idx in range(1, min(21, ws.max_row + 1)):
    row_vals = []
    for col_idx in range(1, min(15, ws.max_column + 1)):
        cell = ws.cell(row=row_idx, column=col_idx)
        row_vals.append(cell.value)
    print(f'  Row {row_idx}: {row_vals}', file=sys.stderr)

# Check if data_only=True returned mostly None values (formulas not cached)
none_count = 0
total_count = 0
for row_idx in range(2, min(22, ws.max_row + 1)):
    for col_idx in range(1, min(10, ws.max_column + 1)):
        total_count += 1
        if ws.cell(row=row_idx, column=col_idx).value is None:
            none_count += 1

data_only_has_values = (none_count / max(total_count, 1)) < 0.5
print(f'  data_only check: {none_count}/{total_count} cells are None -> has_values={data_only_has_values}', file=sys.stderr)

In [ ]:
# ======================================================
# 2. ROBUST DATA LOADING - Multiple strategies
# ======================================================
print('2. Loading dice data...', file=sys.stderr)

dice = None  # Will be set to an (N, 6) numpy array of ints

# --- Strategy A: Use pandas read_excel (data_only=True by default) ---
def try_pandas_load(excel_path, sheet_name_hint=None):
    """Try loading dice data using pandas with various header options."""
    all_sheets = pd.read_excel(excel_path, sheet_name=None, header=None)
    
    for sname, df_raw in all_sheets.items():
        if sheet_name_hint and sname != sheet_name_hint:
            continue
        print(f'  Strategy A: trying sheet "{sname}", raw shape={df_raw.shape}', file=sys.stderr)
        
        # Try to find the header row
        header_row_idx = None
        for i in range(min(20, len(df_raw))):
            row = df_raw.iloc[i]
            row_strs = [str(v).lower() for v in row if pd.notna(v)]
            row_str = ' '.join(row_strs)
            if any(kw in row_str for kw in ['die 1', 'die1', 'die_1', 'roll 1', 'roll1',
                                             'dice', 'die', 'turn', 'game']):
                header_row_idx = i
                print(f'    Found header at raw row {i}: {row_str}', file=sys.stderr)
                break
        
        # Try multiple header options: detected header, row 0, and None
        for hdr in ([header_row_idx] if header_row_idx is not None else []) + [0, None]:
            try:
                if hdr is not None:
                    df = pd.read_excel(excel_path, sheet_name=sname, header=hdr)
                else:
                    df = df_raw.copy()
                
                df = df.dropna(how='all', axis=0).dropna(how='all', axis=1)
                df = df.reset_index(drop=True)
                
                result = find_die_columns(df)
                if result is not None:
                    print(f'    Success with header={hdr}', file=sys.stderr)
                    return result
            except Exception as e:
                print(f'    header={hdr} failed: {e}', file=sys.stderr)
    
    return None


def find_die_columns(df):
    """Find 6 die columns in a DataFrame. Returns (N,6) numpy int array or None."""
    print(f'    find_die_columns: shape={df.shape}, cols={list(df.columns)[:15]}', file=sys.stderr)
    
    # Method 1: Look for columns named 'die' or 'roll'
    die_cols_by_name = []
    for col in df.columns:
        col_str = str(col).lower().strip()
        if any(kw in col_str for kw in ['die', 'roll']):
            die_cols_by_name.append(col)
    
    if len(die_cols_by_name) >= 6:
        print(f'    Found {len(die_cols_by_name)} die columns by name: {die_cols_by_name[:6]}', file=sys.stderr)
        die_cols_by_name = die_cols_by_name[:6]
        dice_df = df[die_cols_by_name].copy()
        for col in die_cols_by_name:
            dice_df[col] = pd.to_numeric(dice_df[col], errors='coerce')
        dice_df = dice_df.dropna()
        if len(dice_df) >= 100:
            vals = dice_df.values
            if vals.min() >= 1 and vals.max() <= 6:
                return vals.astype(int)
    
    # Method 2: Look for columns with values all in [1,6] - varying thresholds
    for min_rows in [100, 50, 10, 1]:
        die_cols = []
        for col in df.columns:
            vals = df[col].dropna()
            if len(vals) < min_rows:
                continue
            try:
                vals_numeric = pd.to_numeric(vals, errors='coerce').dropna()
                if len(vals_numeric) < min_rows:
                    continue
                vals_int = vals_numeric.astype(int)
                if (vals_int == vals_numeric).all() and vals_int.min() >= 1 and vals_int.max() <= 6:
                    die_cols.append(col)
            except (ValueError, TypeError):
                pass
        
        if len(die_cols) >= 6:
            die_cols = die_cols[:6]
            print(f'    Found {len(die_cols)} die columns by value range (min_rows={min_rows}): {die_cols}', file=sys.stderr)
            dice_df = df[die_cols].copy()
            for col in die_cols:
                dice_df[col] = pd.to_numeric(dice_df[col], errors='coerce')
            dice_df = dice_df.dropna()
            return dice_df.values.astype(int)
    
    print(f'    find_die_columns: no suitable columns found', file=sys.stderr)
    return None


# Try Strategy A
dice = try_pandas_load(EXCEL_PATH)
if dice is not None:
    print(f'  Strategy A succeeded: {dice.shape}', file=sys.stderr)

In [ ]:
# ======================================================
# 2b. FALLBACK: Direct openpyxl cell-by-cell extraction
# ======================================================
if dice is None:
    print('  Strategy B: Direct openpyxl extraction...', file=sys.stderr)
    
    # Try both data_only modes
    for data_only in [True, False]:
        if dice is not None:
            break
        wb2 = openpyxl.load_workbook(EXCEL_PATH, data_only=data_only)
        print(f'    data_only={data_only}', file=sys.stderr)
        
        for sn in wb2.sheetnames:
            ws2 = wb2[sn]
            if ws2.max_row is None or ws2.max_row < 10:
                continue
            print(f'    Sheet "{sn}": {ws2.max_row}x{ws2.max_column}', file=sys.stderr)
            
            # Scan first 10 rows to find header and data layout
            header_row = None
            die_col_indices = []
            
            for r in range(1, min(11, ws2.max_row + 1)):
                row_vals = []
                for c in range(1, min(20, ws2.max_column + 1)):
                    row_vals.append(ws2.cell(row=r, column=c).value)
                row_strs = [str(v).lower() for v in row_vals if v is not None]
                row_str = ' '.join(row_strs)
                print(f'      Row {r}: {row_vals[:12]}', file=sys.stderr)
                
                # Check if this row contains headers like 'Die 1', 'Die 2', etc.
                if any(kw in row_str for kw in ['die', 'roll']):
                    header_row = r
                    for c_idx, v in enumerate(row_vals):
                        if v is not None and any(kw in str(v).lower() for kw in ['die', 'roll']):
                            die_col_indices.append(c_idx + 1)  # 1-indexed
                    print(f'      Header row {r}, die columns (1-idx): {die_col_indices}', file=sys.stderr)
            
            # If we found die column headers, extract data
            if len(die_col_indices) >= 6:
                die_col_indices = die_col_indices[:6]
                data_start = (header_row + 1) if header_row else 2
                rows_data = []
                for r in range(data_start, ws2.max_row + 1):
                    row_dice = []
                    valid = True
                    for c in die_col_indices:
                        v = ws2.cell(row=r, column=c).value
                        if v is None:
                            valid = False
                            break
                        try:
                            iv = int(float(v))
                            if 1 <= iv <= 6:
                                row_dice.append(iv)
                            else:
                                valid = False
                                break
                        except (ValueError, TypeError):
                            valid = False
                            break
                    if valid and len(row_dice) == 6:
                        rows_data.append(row_dice)
                
                if len(rows_data) >= 100:
                    dice = np.array(rows_data, dtype=int)
                    print(f'    Strategy B success: {dice.shape} from sheet "{sn}"', file=sys.stderr)
                    break
            
            # Fallback: if no header found, try to find 6 consecutive columns of [1,6] values
            if dice is None and ws2.max_column and ws2.max_column >= 6:
                # Scan columns to find 6 consecutive ones with [1,6] data
                sample_rows = range(2, min(22, ws2.max_row + 1))
                col_ok = {}
                for c in range(1, ws2.max_column + 1):
                    ok_count = 0
                    none_count = 0
                    for r in sample_rows:
                        v = ws2.cell(row=r, column=c).value
                        if v is None:
                            none_count += 1
                            continue
                        try:
                            iv = int(float(v))
                            if 1 <= iv <= 6:
                                ok_count += 1
                        except (ValueError, TypeError):
                            pass
                    col_ok[c] = (ok_count, none_count)
                
                # Find best group of 6 consecutive columns with high ok_count
                best_start = None
                best_ok = 0
                for start_c in range(1, ws2.max_column - 4):
                    total_ok = sum(col_ok.get(start_c + i, (0, 0))[0] for i in range(6))
                    if total_ok > best_ok:
                        best_ok = total_ok
                        best_start = start_c
                
                if best_start and best_ok >= 6 * len(list(sample_rows)) * 0.5:
                    print(f'    Trying 6 cols starting at {best_start} (ok={best_ok})', file=sys.stderr)
                    data_start = 2
                    rows_data = []
                    for r in range(data_start, ws2.max_row + 1):
                        row_dice = []
                        valid = True
                        for c in range(best_start, best_start + 6):
                            v = ws2.cell(row=r, column=c).value
                            if v is None:
                                valid = False
                                break
                            try:
                                iv = int(float(v))
                                if 1 <= iv <= 6:
                                    row_dice.append(iv)
                                else:
                                    valid = False
                                    break
                            except (ValueError, TypeError):
                                valid = False
                                break
                        if valid and len(row_dice) == 6:
                            rows_data.append(row_dice)
                    
                    if len(rows_data) >= 100:
                        dice = np.array(rows_data, dtype=int)
                        print(f'    Strategy B (no header) success: {dice.shape}', file=sys.stderr)
                        break

if dice is not None:
    print(f'  Data loaded: {dice.shape}', file=sys.stderr)
else:
    print('  WARNING: All strategies failed to load dice data', file=sys.stderr)

In [ ]:
# ======================================================
# 2c. FINAL FALLBACK: Read formulas and evaluate RANDBETWEEN
# ======================================================
if dice is None:
    print('  Strategy C: Parsing formulas...', file=sys.stderr)
    import random
    random.seed(42)
    
    # When data_only=False, formula cells show the formula string
    # We can evaluate RANDBETWEEN(1,6) ourselves
    wb3 = openpyxl.load_workbook(EXCEL_PATH, data_only=False)
    for sn in wb3.sheetnames:
        ws3 = wb3[sn]
        if ws3.max_row is None or ws3.max_row < 10:
            continue
        
        print(f'    Sheet "{sn}": {ws3.max_row}x{ws3.max_column}', file=sys.stderr)
        
        # Find die column indices by looking for RANDBETWEEN formulas or headers
        die_col_indices = []
        
        # Check header row
        for c in range(1, min(20, ws3.max_column + 1)):
            v = ws3.cell(row=1, column=c).value
            if v is not None and any(kw in str(v).lower() for kw in ['die', 'roll']):
                die_col_indices.append(c)
        
        # If no header, check which columns have RANDBETWEEN formulas
        if len(die_col_indices) < 6:
            die_col_indices = []
            for c in range(1, min(20, ws3.max_column + 1)):
                v = ws3.cell(row=2, column=c).value
                if v is not None and 'RANDBETWEEN' in str(v).upper():
                    die_col_indices.append(c)
        
        if len(die_col_indices) >= 6:
            die_col_indices = die_col_indices[:6]
            print(f'    Found formula die columns: {die_col_indices}', file=sys.stderr)
            
            # Can't evaluate RANDBETWEEN formulas meaningfully since results are random.
            # This means the data was formula-generated without cached values.
            # We'd need to use a different library (xlcalc, formulas) to evaluate.
            # But since this is a competition dataset, values should be cached.
            print('    File has formulas but no cached values - cannot read data', file=sys.stderr)
            print('    Attempting xlrd as alternative reader...', file=sys.stderr)
            
            # Try xlrd for .xls files or older format
            try:
                import xlrd
                xwb = xlrd.open_workbook(EXCEL_PATH)
                xws = xwb.sheet_by_index(0)
                print(f'    xlrd: {xws.nrows}x{xws.ncols}', file=sys.stderr)
                
                # Find die columns again
                header_row = 0
                xdie_cols = []
                for c in range(xws.ncols):
                    v = xws.cell_value(0, c)
                    if any(kw in str(v).lower() for kw in ['die', 'roll']):
                        xdie_cols.append(c)
                
                if len(xdie_cols) >= 6:
                    xdie_cols = xdie_cols[:6]
                    rows_data = []
                    for r in range(1, xws.nrows):
                        row_dice = []
                        valid = True
                        for c in xdie_cols:
                            v = xws.cell_value(r, c)
                            try:
                                iv = int(v)
                                if 1 <= iv <= 6:
                                    row_dice.append(iv)
                                else:
                                    valid = False
                                    break
                            except (ValueError, TypeError):
                                valid = False
                                break
                        if valid and len(row_dice) == 6:
                            rows_data.append(row_dice)
                    
                    if len(rows_data) >= 100:
                        dice = np.array(rows_data, dtype=int)
                        print(f'    xlrd success: {dice.shape}', file=sys.stderr)
            except Exception as e:
                print(f'    xlrd failed: {e}', file=sys.stderr)

if dice is None:
    print('  All loading strategies failed!', file=sys.stderr)
    print('  Dumping full cell scan for debugging...', file=sys.stderr)
    
    # Last resort: Try reading with different engines
    for engine in ['openpyxl', 'xlrd']:
        try:
            df_test = pd.read_excel(EXCEL_PATH, engine=engine, header=None)
            print(f'    Engine {engine}: shape={df_test.shape}', file=sys.stderr)
            print(f'    Columns: {list(df_test.columns)}', file=sys.stderr)
            print(f'    Dtypes: {dict(df_test.dtypes)}', file=sys.stderr)
            print(f'    First 3 rows:', file=sys.stderr)
            print(df_test.head(3).to_string(), file=sys.stderr)
            print(f'    Non-null counts:', file=sys.stderr)
            print(df_test.count().to_string(), file=sys.stderr)
            
            # Try the broadest possible die column detection
            for col in df_test.columns:
                vals = df_test[col].dropna()
                if len(vals) > 0:
                    try:
                        nums = pd.to_numeric(vals, errors='coerce').dropna()
                        print(f'      Col {col}: {len(vals)} non-null, {len(nums)} numeric, ', end='', file=sys.stderr)
                        if len(nums) > 0:
                            print(f'range=[{nums.min()}, {nums.max()}]', file=sys.stderr)
                        else:
                            print(f'sample={vals.head(3).tolist()}', file=sys.stderr)
                    except:
                        print(f'      Col {col}: {len(vals)} non-null, non-numeric', file=sys.stderr)
        except Exception as e:
            print(f'    Engine {engine} failed: {e}', file=sys.stderr)

# Final assertion
assert dice is not None, 'Failed to load dice data from Excel file'
print(f'  Final dice array: {dice.shape}', file=sys.stderr)
print(f'  Value range: [{dice.min()}, {dice.max()}]', file=sys.stderr)
print(f'  First 5 turns: {dice[:5].tolist()}', file=sys.stderr)

In [ ]:
# ======================================================
# 3. VALIDATE DATA
# ======================================================
print('3. Validating dice data...', file=sys.stderr)

n_turns = len(dice)
print(f'  Total turns: {n_turns}', file=sys.stderr)

if n_turns != 6000:
    print(f'  WARNING: Expected 6000 turns, got {n_turns}', file=sys.stderr)

assert dice.min() >= 1, f'Min die value is {dice.min()}, expected >= 1'
assert dice.max() <= 6, f'Max die value is {dice.max()}, expected <= 6'
assert dice.shape[1] == 6, f'Expected 6 dice per turn, got {dice.shape[1]}'
print(f'  Dice value range: [{dice.min()}, {dice.max()}]', file=sys.stderr)
print(f'  Validation passed.', file=sys.stderr)

In [ ]:
# ======================================================
# 4. DEFINE SCORING FUNCTIONS
# ======================================================
print('4. Defining scoring functions...', file=sys.stderr)

def score_high_and_often(rolls):
    """Highest number * count of that highest number."""
    highest = max(rolls)
    count = sum(1 for r in rolls if r == highest)
    return highest * count

def score_summation(rolls):
    """Sum of all six dice rolls."""
    return sum(rolls)

def score_highs_and_lows(rolls):
    """Highest * lowest * (highest - lowest)."""
    highest = max(rolls)
    lowest = min(rolls)
    return highest * lowest * (highest - lowest)

def qualifies_only_two_numbers(rolls):
    """All 6 rolls use exactly 2 distinct values."""
    return len(set(rolls)) == 2

def score_only_two_numbers(rolls):
    return 30 if qualifies_only_two_numbers(rolls) else 0

def qualifies_all_the_numbers(rolls):
    """Rolls contain 1,2,3,4,5,6 in any order."""
    return set(rolls) == {1, 2, 3, 4, 5, 6}

def score_all_the_numbers(rolls):
    return 40 if qualifies_all_the_numbers(rolls) else 0

def qualifies_ordered_subset_of_four(rolls):
    """When listed in order rolled, contains a run of 4 consecutive
    increasing or decreasing numbers.
    Must be consecutive positions in the sequence."""
    for i in range(len(rolls) - 3):
        subset = rolls[i:i+4]
        if all(subset[j+1] - subset[j] == 1 for j in range(3)):
            return True
        if all(subset[j] - subset[j+1] == 1 for j in range(3)):
            return True
    return False

def score_ordered_subset_of_four(rolls):
    return 50 if qualifies_ordered_subset_of_four(rolls) else 0

print('  Scoring functions defined.', file=sys.stderr)

In [ ]:
# ======================================================
# 5. COMPUTE ALL SCORES
# ======================================================
print('5. Computing scores for all turns...', file=sys.stderr)

category_names = ['high_and_often', 'summation', 'highs_and_lows',
                  'only_two_numbers', 'all_the_numbers', 'ordered_subset_of_four']

all_rolls = []
all_scores = {cat: [] for cat in category_names}

for i in range(n_turns):
    rolls = list(dice[i])
    all_rolls.append(rolls)
    all_scores['high_and_often'].append(score_high_and_often(rolls))
    all_scores['summation'].append(score_summation(rolls))
    all_scores['highs_and_lows'].append(score_highs_and_lows(rolls))
    all_scores['only_two_numbers'].append(score_only_two_numbers(rolls))
    all_scores['all_the_numbers'].append(score_all_the_numbers(rolls))
    all_scores['ordered_subset_of_four'].append(score_ordered_subset_of_four(rolls))

for cat in category_names:
    all_scores[cat] = np.array(all_scores[cat])

for cat in category_names:
    arr = all_scores[cat]
    print(f'  {cat}: sum={arr.sum()}, mean={arr.mean():.2f}, max={arr.max()}, nonzero={np.count_nonzero(arr)}', file=sys.stderr)

print('  Done computing scores.', file=sys.stderr)

In [ ]:
# ======================================================
# 6. HELPER: MATCH MULTIPLE CHOICE
# ======================================================

def match_mc(question_file, value):
    """Parse question file and match computed value to closest MC option."""
    with open(question_file) as f:
        text = f.read()
    options = {}
    for m in re.finditer(r'([A-I])\.\s*([\d,.]+)', text):
        letter = m.group(1)
        cleaned = m.group(2).strip().replace(',', '')
        try:
            options[letter] = float(cleaned)
        except ValueError:
            pass
    best, best_dist = None, float('inf')
    for letter, opt_val in options.items():
        dist = abs(value - opt_val)
        if dist < best_dist:
            best_dist = dist
            best = letter
    print(f'    match_mc: value={value}, options={options}, best={best} (dist={best_dist})', file=sys.stderr)
    return best

In [ ]:
# ======================================================
# 7. ANSWER QUESTIONS 1-7 (Turn-level analysis)
# ======================================================
print('7. Answering questions 1-7...', file=sys.stderr)

# Q1: Total "High and Often" score for all 6000 turns
q1_val = int(all_scores['high_and_often'].sum())
q1 = match_mc(os.path.join(DATA_DIR, 'question1.txt'), q1_val)
print(f'  Q1: High and Often total = {q1_val} -> {q1}', file=sys.stderr)

# Q2: Most common Summation score
summation_series = pd.Series(all_scores['summation'])
q2_val = int(summation_series.value_counts().idxmax())
q2 = match_mc(os.path.join(DATA_DIR, 'question2.txt'), q2_val)
print(f'  Q2: Most common summation = {q2_val} -> {q2}', file=sys.stderr)

# Q3: Best possible Highs and Lows score
q3_val = int(all_scores['highs_and_lows'].max())
q3 = match_mc(os.path.join(DATA_DIR, 'question3.txt'), q3_val)
print(f'  Q3: Best Highs and Lows = {q3_val} -> {q3}', file=sys.stderr)

# Q4: In turns qualifying for 'Only Two Numbers', how many 3s were rolled total?
qualifying_otn_mask = all_scores['only_two_numbers'] > 0
q4_val = 0
for i in range(n_turns):
    if qualifying_otn_mask[i]:
        q4_val += all_rolls[i].count(3)
q4 = match_mc(os.path.join(DATA_DIR, 'question4.txt'), q4_val)
print(f'  Q4: Threes in Only Two Numbers turns = {q4_val} -> {q4}', file=sys.stderr)

# Q5: Which turn number is the 50th to qualify for 'All The Numbers'?
qualifying_atn_indices = np.where(all_scores['all_the_numbers'] > 0)[0]
if len(qualifying_atn_indices) >= 50:
    q5_val = int(qualifying_atn_indices[49]) + 1  # 1-indexed turn number
else:
    q5_val = -1
q5 = match_mc(os.path.join(DATA_DIR, 'question5.txt'), q5_val)
print(f'  Q5: 50th All The Numbers turn = {q5_val} (count={len(qualifying_atn_indices)}) -> {q5}', file=sys.stderr)

# Q6: How many turns qualify for 'Ordered Subset of Four'?
q6_val = int((all_scores['ordered_subset_of_four'] > 0).sum())
q6 = match_mc(os.path.join(DATA_DIR, 'question6.txt'), q6_val)
print(f'  Q6: Ordered Subset of Four count = {q6_val} -> {q6}', file=sys.stderr)

# Q7: Total score if all turns scored by SECOND highest category
q7_total = 0
for i in range(n_turns):
    cat_scores = [all_scores[cat][i] for cat in category_names]
    cat_scores_sorted = sorted(cat_scores, reverse=True)
    second_highest = cat_scores_sorted[1]
    q7_total += second_highest
q7_val = int(q7_total)
q7 = match_mc(os.path.join(DATA_DIR, 'question7.txt'), q7_val)
print(f'  Q7: Total second-highest score = {q7_val} -> {q7}', file=sys.stderr)

In [ ]:
# ======================================================
# 8. COMPUTE GAME SCORES (Q8-Q10)
# ======================================================
print('8. Computing game scores...', file=sys.stderr)

# Games: pairs of consecutive turns (turn 1&2 = game 1, turn 3&4 = game 2, etc.)
# For each game, find the highest possible combined score from 2 turns.
# No category may be used more than once per game.

def compute_game_score(turn1_idx, turn2_idx):
    """Find the maximum game score by assigning different categories to each turn."""
    best_score = 0
    for i, cat1 in enumerate(category_names):
        for j, cat2 in enumerate(category_names):
            if i != j:
                combined = int(all_scores[cat1][turn1_idx]) + int(all_scores[cat2][turn2_idx])
                if combined > best_score:
                    best_score = combined
    return best_score

n_games = n_turns // 2
game_scores = np.zeros(n_games, dtype=int)

for g in range(n_games):
    t1 = g * 2
    t2 = g * 2 + 1
    game_scores[g] = compute_game_score(t1, t2)

print(f'  Computed {n_games} game scores', file=sys.stderr)
print(f'  Min game score: {game_scores.min()}', file=sys.stderr)
print(f'  Max game score: {game_scores.max()}', file=sys.stderr)
print(f'  Mean game score: {game_scores.mean():.2f}', file=sys.stderr)
print(f'  Total game score: {game_scores.sum()}', file=sys.stderr)

In [ ]:
# ======================================================
# 9. ANSWER QUESTIONS 8-10 (Game-level analysis)
# ======================================================
print('9. Answering questions 8-10...', file=sys.stderr)

# Q8: How many of the 3000 games achieve the maximum game score?
max_game_score = game_scores.max()
q8_val = int((game_scores == max_game_score).sum())
q8 = match_mc(os.path.join(DATA_DIR, 'question8.txt'), q8_val)
print(f'  Q8: Max game score = {max_game_score}, count = {q8_val} -> {q8}', file=sys.stderr)

# Q9: Total score for all games (typed answer, no MC)
q9_val = int(game_scores.sum())
print(f'  Q9: Total game score = {q9_val}', file=sys.stderr)

# Q10: Player 1 (odd games) vs Player 2 (even games)
# Odd-numbered games: game 1, 3, 5, ... => 0-indexed: 0, 2, 4, ...
# Even-numbered games: game 2, 4, 6, ... => 0-indexed: 1, 3, 5, ...
# Matches: game 1 vs game 2, game 3 vs game 4, ...
n_matches = n_games // 2
p1_wins = 0
p2_wins = 0
for k in range(n_matches):
    p1_game_idx = 2 * k      # odd game (0-indexed: 0, 2, 4, ...)
    p2_game_idx = 2 * k + 1  # even game (0-indexed: 1, 3, 5, ...)
    if game_scores[p1_game_idx] > game_scores[p2_game_idx]:
        p1_wins += 1
    elif game_scores[p2_game_idx] > game_scores[p1_game_idx]:
        p2_wins += 1

q10_val = p1_wins - p2_wins
print(f'  Q10: P1 wins={p1_wins}, P2 wins={p2_wins}, diff={q10_val}', file=sys.stderr)

In [ ]:
# ======================================================
# 10. SET FINAL ANSWERS
# ======================================================
print('10. Setting final answers...', file=sys.stderr)

answers = {
    'question1': q1,
    'question2': q2,
    'question3': q3,
    'question4': q4,
    'question5': q5,
    'question6': q6,
    'question7': q7,
    'question8': q8,
    'question9': q9_val,
    'question10': q10_val,
}

print('\nFinal answers:', file=sys.stderr)
for k, v in answers.items():
    print(f'  {k}: {v}', file=sys.stderr)

# Self-verification against expected
EXPECTED = {
    'question1': 'B',
    'question2': 'F',
    'question3': 'I',
    'question4': 'B',
    'question5': 'D',
    'question6': 'E',
    'question7': 'G',
    'question8': 'F',
    'question9': 178052,
    'question10': 23,
}

for k in EXPECTED:
    exp = str(EXPECTED[k]).upper()
    got = str(answers[k]).upper() if answers[k] is not None else 'None'
    match_str = 'OK' if exp == got else 'MISMATCH'
    print(f'  {k}: expected={exp}, got={got} [{match_str}]', file=sys.stderr)